In [1]:
# 1. Mira con qué cuenta real se está ejecutando tu Jupyter
!gcloud config get-value account

# 2. Lista todos los buckets a los que esa cuenta SÍ tiene acceso real
!gsutil ls


302885999194-compute@developer.gserviceaccount.com
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://li02_nhc_dsua_buckito2026/


In [2]:
import io
import pandas as pd
import yaml
from google.cloud import storage
from google.api_core import exceptions

# ---- SOLUCIÓN PARA LA ETIQUETA !expr ----
# Creamos un constructor personalizado para que PyYAML entienda '!expr' como texto
def expr_constructor(loader, node):
    return loader.construct_scalar(node)

# Registramos el constructor en el cargador seguro de PyYAML
yaml.SafeLoader.add_constructor('!expr', expr_constructor)
# ------------------------------------------

# 1. Configuración de nombres de buckets y prefijos
BUCKETS = [
    "li01_nhc_dsua_buckito2026",
    "li02_nhc_dsua_buckito2026"
]
PREFIX_EXP = "exp/"
ALLOWED_PREFIXES = (
    "exp/WFz635_WorkFlow_01_senior_NHC",
    "exp/WFz635_WorkFlow_01_senior_NHC_B"
)

# Inicializa el cliente de GCP Storage
client = storage.Client()
all_data = []

for bucket_name in BUCKETS:
    print(f"Procesando bucket: {bucket_name}...")
    bucket = client.bucket(bucket_name)
    
    try:
        blobs = bucket.list_blobs(prefix=PREFIX_EXP)
        
        # Validación temprana de acceso/existencia
        blobs_iterator = iter(blobs)
        first_blob = next(blobs_iterator, None)
        
        if first_blob is None:
            print(f" -> Advertencia: No se encontraron archivos o el prefijo '{PREFIX_EXP}' no existe en {bucket_name}.\n")
            continue
            
        all_blobs = [first_blob] + list(blobs_iterator)
        
    except exceptions.Forbidden:
        print(f" -> OMITIDO: No tienes permisos (403 Forbidden) para acceder al bucket '{bucket_name}'.\n")
        continue
    except exceptions.NotFound:
        print(f" -> OMITIDO: El bucket '{bucket_name}' no existe o no fue encontrado (404 Not Found).\n")
        continue
    except Exception as e:
        print(f" -> OMITIDO: Error inesperado al acceder al bucket '{bucket_name}': {e}\n")
        continue
    
    # Procesar los archivos PARAM.yml
    for blob in all_blobs:
        if not blob.name.endswith("/PARAM.yml"):
            continue
            
        if not blob.name.startswith(ALLOWED_PREFIXES):
            continue
        
        try:
            parts = blob.name.split('/')
            experiment_name = parts[2] if len(parts) > 2 else "unknown"
            
            yaml_content = blob.download_as_text()
            
            # Ahora usamos safe_load con la configuración que acepta '!expr'
            param_dict = yaml.safe_load(yaml_content)
            
            if isinstance(param_dict, dict):
                param_dict["_bucket_origen"] = bucket_name
                param_dict["_experimento"] = experiment_name
                all_data.append(param_dict)
                print(f" -> Procesado con éxito: {experiment_name}")
            else:
                print(f" -> Advertencia: {blob.name} no tiene un formato de diccionario válido.")
                
        except Exception as e:
            print(f" -> Error al procesar {blob.name}: {e}")
    print() 

# 2. Consolidar en un DataFrame
if all_data:
    df = pd.DataFrame(all_data)
    
    # Reorganizar columnas
    cols = ["_bucket_origen", "_experimento"] + [c for c in df.columns if c not in ["_bucket_origen", "_experimento"]]
    df = df[cols]
    
    # Guardar en archivo
    output_file = "consolidado_experimentos_li02_nhc_dsua.csv"
    df.to_csv(output_file, index=False, encoding="utf-8")
    print(f"¡Proceso terminado! Datos guardados en '{output_file}' con {len(df)} filas.")
else:
    df = pd.DataFrame() 
    print("No se pudo extraer información de ningún archivo PARAM.yml.")



Procesando bucket: li01_nhc_dsua_buckito2026...
 -> OMITIDO: El bucket 'li01_nhc_dsua_buckito2026' no existe o no fue encontrado (404 Not Found).

Procesando bucket: li02_nhc_dsua_buckito2026...
 -> Procesado con éxito: PARAM.yml
 -> Procesado con éxito: PARAM.yml
 -> Procesado con éxito: PARAM.yml
 -> Procesado con éxito: PARAM.yml
 -> Procesado con éxito: PARAM.yml
 -> Procesado con éxito: PARAM.yml
 -> Procesado con éxito: PARAM.yml
 -> Procesado con éxito: PARAM.yml
 -> Procesado con éxito: PARAM.yml
 -> Procesado con éxito: PARAM.yml
 -> Procesado con éxito: PARAM.yml

¡Proceso terminado! Datos guardados en 'consolidado_experimentos_li02_nhc_dsua.csv' con 11 filas.
